In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr
import cartopy.crs as ccrs
import cartopy.feature as cfeature

In [ ]:
#=============== 1. 평균 — mean(dim=...) 
# mean(dim=)은 그 차원을 "없애면서" 평균함
#   (time, level, lat, lon) 에서 dim='lon' 을 평균 → (time, level, lat)  : 동서평균
#   (time, lat, lon)        에서 dim=['lat','lon'] 평균 → (time,)       : 전구평균 시계열

In [ ]:
#-------- 동서평균 (zonal mean) : 바람 자료 --------
var  = xr.open_dataset('../data/uwnd.mon.mean.nc')
uwnd = var['uwnd']
print(uwnd)                          # (time, level, lat, lon) — 과제에서 쓸 파일이 이렇게 생김

In [ ]:
zmean = uwnd.mean(dim='lon')         # lon 이 사라짐 → (time, level, lat)
print(zmean.shape)

In [ ]:
print(zmean['level']) #1000 hPa ≈ 해수면(0 km)에서 10 hPa ≈ 약 31 km까지 

In [ ]:
zmean.sel(time='2010-01-01').plot.contourf(levels=50, cmap='jet')   # 남은 (level, lat) 를 단면으로
plt.gca().invert_yaxis()             # 기압은 위로 갈수록 작아지므로 y축 뒤집기
plt.title('Zonal mean U (2010-01)')
plt.show()

In [ ]:
#-------- 전구평균 시계열 : SST -------- 
var = xr.open_dataset('../data/sst.mnmean.nc')
sst = var['sst']
mean = sst.mean(dim=['lat', 'lon'])  # lat, lon 이 사라짐 → (time,)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4), dpi=120)
ax.plot(sst['time'].values, mean, color='black')
ax.set_xlabel('Time'); ax.set_ylabel('[$\\degree$C]'); ax.set_title('Global Mean SST')
plt.show()

In [ ]:
#-------- 가중평균 (weighted mean) --------  
# 위경도 격자는 고위도로 갈수록 실제 면적이 작아짐 → 그냥 평균하면 극지방이 과대평가됨
# 면적은 cos(위도)에 비례하므로 이걸 가중치로 씀
weights = np.cos(np.deg2rad(sst['lat']))
weights.name = 'weights'

wmean = sst.weighted(weights).mean(dim=['lat', 'lon'])


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4), dpi=120)
ax.plot(sst['time'].values, mean,  color='black', label='simple mean')
ax.plot(sst['time'].values, wmean, color='red',   label='weighted mean')
ax.set_xlabel('Time')
ax.set_ylabel('[$\\degree$C]')
ax.set_title('Global Mean SST')
ax.legend()
plt.show()

In [ ]:
#=============== 2. 기후값 (climatology) — groupby [실습] ===============
# groupby('time.month') : 시간축을 1월끼리, 2월끼리, ... 12묶음으로 나눔
# .mean(dim='time')      : 각 묶음 안에서 평균 → 결과는 time 대신 month 차원 (12개)
# sel(time=slice(...))   : 기준 기간을 먼저 자름 (기후값은 보통 30년 기준)
dset = xr.open_dataset('../data/sst.mnmean.nc')
sst  = dset['sst']
clim = sst.sel(time=slice('1982-01', '2022-12')).groupby('time.month').mean(dim='time')
print(clim)                          # (month, lat, lon) 인지 확인
# 'time.season' 으로 묶으면 계절별 (DJF, MAM, JJA, SON) — 과제에서 씀
# clim_season = sst.groupby('time.season').mean(dim='time')

In [ ]:
#-------- 기후값 지도 4장 : 1월, 4월, 7월, 10월 --------  
# subplot(2, 2, n) : 2행 2열 중 n번 (1부터)


#------------- Plotting a Map---------------
fig = plt.figure(figsize=(18,14), dpi=100)

clevs = np.arange(-3, 33, 1)   # Set color levels

ax1 = plt.subplot(221, projection = ccrs.Robinson(180))
a = clim.isel(month=0).plot.contourf(ax=ax1, levels = clevs, cmap='jet', transform=ccrs.PlateCarree(),
      cbar_kwargs={'extendrect': 'True', 'orientation': 'horizontal', 'pad': 0.06, 'aspect': 30}) 
ax1.set_title('SST Climatology (JAN)',fontsize=17)
ax1.coastlines()
ax1.add_feature(cfeature.NaturalEarthFeature('physical', 'land', '50m', edgecolor='face', facecolor='black'))
ax1.gridlines(draw_labels=True, linewidth=1, linestyle=':', color='gray', x_inline=False)


# Sub-Figure 2
ax2 = plt.subplot(222, projection = ccrs.Robinson(180))
b = clim.isel(month=3).plot.contourf(ax=ax2, levels = clevs, cmap='jet', transform=ccrs.PlateCarree(),
    cbar_kwargs={'extendrect': 'True', 'orientation': 'horizontal', 'pad': 0.06, 'aspect': 30}) 
ax2.set_title('SST Climatology (APR)',fontsize=17)
ax2.coastlines()
ax2.add_feature(cfeature.NaturalEarthFeature('physical', 'land', '10m', edgecolor='face', facecolor='black'))
ax2.gridlines(draw_labels=True, linewidth=1, linestyle=':', color='gray')

# Sub-Figure 3
ax3 = plt.subplot(223, projection = ccrs.Robinson(180))
c = clim.isel(month=6).plot.contourf(ax=ax3, levels = clevs, cmap='jet', transform=ccrs.PlateCarree(),
    cbar_kwargs={'extendrect': 'True', 'orientation': 'horizontal', 'pad': 0.06, 'aspect': 30}) 
ax3.set_title('SST Climatology (JUL)',fontsize=17)
ax3.coastlines()
ax3.add_feature(cfeature.NaturalEarthFeature('physical', 'land', '50m', edgecolor='face', facecolor='black'))
ax3.gridlines(draw_labels=True, linewidth=1, linestyle=':', color='gray')

# Sub-Figure 4

ax4 = plt.subplot(224, projection = ccrs.Robinson(180))
d = clim.isel(month=9).plot.contourf(ax=ax4, levels = clevs, cmap='jet', transform=ccrs.PlateCarree(),
    cbar_kwargs={'extendrect': 'True', 'orientation': 'horizontal', 'pad': 0.06, 'aspect': 30}) 
ax4.set_title('SST Climatology (OCT)',fontsize=17)
ax4.coastlines()
ax4.add_feature(cfeature.NaturalEarthFeature('physical', 'land', '50m', edgecolor='face', facecolor='black'))
ax4.gridlines(draw_labels=True, linewidth=1, linestyle=':', color='gray')


plt.savefig('SST_monthly_climatology.png')
plt.show()

In [ ]:
#=============== 3. 아노말리 (anomaly) ===============
# 아노말리 = 관측값 − 그 달의 기후값
# sst.groupby('time.month') - clim  
anom = sst.groupby('time.month') - clim
print(anom)                          # 차원은 sst 와 같음 (time, lat, lon)

In [ ]:

# 한 시점 확인
anom.sel(time='2015-12-01').plot(cmap='RdBu_r', vmin=-4, vmax=4)   # 2015/16 엘니뇨
plt.show()

In [ ]:
#=============== 4. Nino 3.4 지수 ===============
# where(조건, drop=True) : 조건에 맞는 격자만 남기고 나머지는 버림
# Nino3.4 영역 : 5S~5N, 170W~120W (경도 0~360 자료에서는 190~240)
nino34area = anom.where((anom.lat < 5) & (anom.lat > -5) &
                        (anom.lon > 190) & (anom.lon < 240), drop=True)
nino34 = nino34area.mean(dim=['lat', 'lon'])        # 영역 평균 → (time,)

# rolling(time=3, center=True) : 앞뒤 한 달씩 포함한 3개월 이동평균
nino3mon = nino34.rolling(time=3, center=True).mean()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4), dpi=120)
t = nino3mon['time'].values
ax.fill_between(t, nino3mon.where(nino3mon >= 0).values, 0, color='red',  alpha=0.8)   # 양수 구간 빨강
ax.fill_between(t, nino3mon.where(nino3mon <= 0).values, 0, color='blue', alpha=0.8)   # 음수 구간 파랑
ax.plot(t, nino3mon, color='black', linewidth=0.8)
ax.axhline(1.0,color='black',linewidth=0.5,linestyle='dashed')
ax.axhline(-1.0,color='black',linewidth=0.5,linestyle='dashed')
ax.axhline(1.5,color='black',linewidth=0.6,linestyle='dashed')
ax.axhline(-1.5,color='black',linewidth=0.6,linestyle='dashed')
ax.axhline(2.0,color='black',linewidth=0.7,linestyle='dashed')
ax.axhline(-2.0,color='black',linewidth=0.7,linestyle='dashed')
ax.axhline(0, color='k', linewidth=0.5)
ax.set_xlabel('Time'); ax.set_ylabel('[$\\degree$C]'); ax.set_title('Nino 3.4 index')
plt.savefig('nino34_index.png')
plt.show()

In [ ]:
#-------- 계절(DJF)만 뽑기 -------- 
# 월별 자료에서 [start::12] 는 12달마다 하나 → 특정 달만 뽑힘
# 3개월 이동평균에서 1월을 뽑으면 = 12·1·2월 평균 = DJF
ninos = nino3mon.sel(time=slice('1981-12', '2022-12'))[1::12]   # 1981-12 부터 시작하므로 [1] = 1982-01
print(ninos['time'].values[:3])                                  # 정말 1월인지 확인
print(len(ninos))                                                # 몇 년치인지       


In [ ]:
# barplot을 그릴 때 ninos의 총 value 개수와 x를 맞추어야 함 

# x축 연도 — 둘 중 하나
# ax.bar(list(range(1982, 2023)), ninos, ...)       # 손으로 세기: 개수가 len(ninos)와 같아야 함 (다르면 에러)
# ax.bar(ninos['time'].dt.year.values, ninos, ...)  # 자료에서 꺼내기: 자료가 바뀌어도 항상 맞음 *그러나 연도를 뽑아와야한다는 것에 주의! 

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4), dpi=120)
ax.bar(list(range(1982, 2023)), ninos, color='g', width=0.72) #1

ax.axhline(1.0,color='black',linewidth=0.5,linestyle='dashed')
ax.axhline(-1.0,color='black',linewidth=0.5,linestyle='dashed')
ax.axhline(1.5,color='black',linewidth=0.6,linestyle='dashed')
ax.axhline(-1.5,color='black',linewidth=0.6,linestyle='dashed')
ax.axhline(2.0,color='black',linewidth=0.7,linestyle='dashed')
ax.axhline(-2.0,color='black',linewidth=0.7,linestyle='dashed')

ax.axhline(0, color='k', linewidth=0.5)
ax.axhline(ninos.std().values, color='red', linewidth=0.5, linestyle='dashed')   # 표준편차 기준선
ax.set_xlabel('Year'); ax.set_ylabel('[$\\degree$C]'); ax.set_title('Nino 3.4 index (DJF)')
plt.show()